# Initial EDA: Playground Series S6E7

This notebook is meant to do three things:

1. Help me understand the dataset before changing the baseline
2. Surface class imbalance, missingness, and feature patterns
3. Create talking points I can explain clearly in an interview


## Questions To Answer

- What does the target distribution look like?
- How much missing data do we have?
- Which numeric features separate the classes most clearly?
- How do the categorical features differ by class?
- What does this suggest about the next modeling iteration?


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


In [ ]:
PROJECT_DIR = Path.cwd().resolve().parents[0]
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"

train_df = pd.read_csv(RAW_DATA_DIR / "train.csv")
test_df = pd.read_csv(RAW_DATA_DIR / "test.csv")
sample_submission_df = pd.read_csv(RAW_DATA_DIR / "sample_submission.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Sample submission shape:", sample_submission_df.shape)

In [ ]:
train_df.head()

In [ ]:
train_df.info()

## Target Distribution

This matters because the competition metric is balanced accuracy, which is usually a clue that plain accuracy can be misleading.

In [ ]:
target_counts = train_df["health_condition"].value_counts()
target_pct = (train_df["health_condition"].value_counts(normalize=True) * 100).round(2)
display(pd.DataFrame({"count": target_counts, "percent": target_pct}))

plt.figure(figsize=(8, 4))
sns.countplot(data=train_df, x="health_condition", order=target_counts.index, palette="deep")
plt.title("Target Distribution")
plt.xlabel("Health Condition")
plt.ylabel("Count")
plt.show()

## Missing Data Profile

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": train_df.isnull().sum(),
    "missing_percent": (train_df.isnull().mean() * 100).round(2),
}).sort_values("missing_percent", ascending=False)

missing_summary

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(
    data=missing_summary.reset_index().rename(columns={"index": "feature"}),
    x="missing_percent",
    y="feature",
    palette="crest",
)
plt.title("Percent Missing By Feature")
plt.xlabel("Missing Percent")
plt.ylabel("Feature")
plt.show()

## Numeric Feature Overview

In [ ]:
numeric_columns = train_df.select_dtypes(include=["number"]).columns.tolist()
numeric_columns.remove("id")
numeric_columns

In [ ]:
train_df.groupby("health_condition")[numeric_columns].mean().round(2)

In [ ]:
for column in numeric_columns:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=train_df.sample(min(len(train_df), 50000), random_state=42), x="health_condition", y=column, palette="deep")
    plt.title(f"{column} by health_condition")
    plt.show()

## Categorical Feature Overview

In [ ]:
categorical_columns = train_df.select_dtypes(include=["object"]).columns.tolist()
categorical_columns.remove("health_condition")
categorical_columns

In [ ]:
for column in categorical_columns:
    display(pd.crosstab(train_df[column], train_df["health_condition"], normalize="index").round(3))

In [ ]:
for column in categorical_columns:
    plt.figure(figsize=(9, 4))
    plot_df = train_df[column].fillna("missing").to_frame().assign(health_condition=train_df["health_condition"])
    sns.countplot(data=plot_df, x=column, hue="health_condition", palette="deep")
    plt.title(f"{column} by health_condition")
    plt.xticks(rotation=20)
    plt.show()

## Quick Comparison: Train vs Test Missingness

This is a useful sanity check so we can spot any train/test drift in missing-value patterns.

In [ ]:
train_missing = train_df.isnull().mean().rename("train_missing_pct") * 100
test_missing = test_df.isnull().mean().rename("test_missing_pct") * 100
pd.concat([train_missing, test_missing], axis=1).round(2).sort_values("train_missing_pct", ascending=False)

## Sample Rows By Class

Seeing actual rows helps translate abstract features into something more intuitive.

In [ ]:
for label in ["fit", "at-risk", "unhealthy"]:
    print(f"\n--- {label} ---")
    display(train_df[train_df["health_condition"] == label].head(5))

## Working Notes

Use this section to write down what seems predictive, suspicious, or worth trying next.

In [ ]:
# Notes:
# 1.
# 2.
# 3.
